In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
df.shape

(2000, 8)

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [6]:
df=df.drop('id',axis=1)

In [7]:
answer=df['answer']
df=df.drop('answer',axis=1)

In [8]:
df.shape

(2000, 6)

In [9]:
import re

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r'[^a-zA-Z0-9\s]',' ',text)
    text=re.sub(r'\s+',' ',text).strip()
    return text

df['prompt']=df['prompt'].apply(clean_text)
df['A']=df['A'].apply(clean_text)
df['B']=df['B'].apply(clean_text)
df['C']=df['C'].apply(clean_text)
df['D']=df['D'].apply(clean_text)
df['E']=df['E'].apply(clean_text)

In [10]:
from sklearn.model_selection import train_test_split

x_train,x_val,y_train,y_val=train_test_split(df,answer,train_size=0.8,random_state=1)

In [11]:
def score_to_labels(scores,top1=False):
    labels=[]
    top1_labels=[]
    for score in scores:
        scored=np.argsort(score)[::-1][:3]
        top_3=[chr(65+i) for i in scored]
        labels.append(" ".join(top_3))
        top1_labels.append(chr(65+np.argsort(score)[::-1][0]))
    if not top1:
        return labels
    else:
        return top1_labels

In [12]:
def map_at_3(predictions,ground_truth):
    ap_scores=[]
    for pred, truth in zip(predictions, ground_truth):
        pred_list = pred.split()
        if truth in pred_list:
            ap_scores.append(1.0 / (pred_list.index(truth) + 1))
        else:
            ap_scores.append(0.0)
    return sum(ap_scores) / len(ap_scores)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True)
all_text=df['prompt'].tolist()+df['A'].tolist()+df['B'].tolist()+df['C'].tolist()+df['D'].tolist()+df['E'].tolist()
tfidf.fit(all_text)

TfidfVectorizer(sublinear_tf=True)

In [14]:
tfidf_train_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_train['prompt']+" "+x_train[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_train_scores.append(np.asarray(scores).flatten())
tfidf_train_scores=np.array(tfidf_train_scores).T

In [15]:
tfidf_val_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_val['prompt']+" "+x_val[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_val_scores.append(np.asarray(scores).flatten())
tfidf_val_scores=np.array(tfidf_val_scores).T

In [16]:
tfidf_train_labels=score_to_labels(tfidf_train_scores,top1=False)
tfidf_train_map3=map_at_3(tfidf_train_labels,y_train)
print(tfidf_train_map3)

tfidf_val_labels=score_to_labels(tfidf_val_scores,top1=False)
tfidf_val_map3=map_at_3(tfidf_val_labels,y_val)
print(tfidf_val_map3)

0.524375
0.5133333333333333


In [17]:
from sklearn.metrics import accuracy_score,f1_score

def support_metrics(train_scores,val_scores):
    train_acc=accuracy_score(score_to_labels(train_scores,top1=True),y_train)
    val_acc=accuracy_score(score_to_labels(val_scores,top1=True),y_val)
    train_f1=f1_score(score_to_labels(train_scores,top1=True),y_train,average='macro')
    val_f1=f1_score(score_to_labels(val_scores,top1=True),y_val,average='macro')
    return train_acc,val_acc,train_f1,val_f1
    
tfidf_train_acc,tfidf_val_acc,tfidf_train_f1,tfidf_val_f1=support_metrics(tfidf_train_scores,tfidf_val_scores)

In [18]:
!pip install wandb -q

In [19]:
import wandb
from kaggle_secrets import UserSecretsClient
import os

secrets=UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login()
wandb_mode=True

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [20]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="tfidf-baseline-metrics",id="tfidf_v1",resume="allow")
    wandb.log({
        "map_at_3_train":tfidf_train_map3,   
        "map_at_3_val":tfidf_val_map3,
        "accuracy_at_train":tfidf_train_acc,
        "accuracy_at_val":tfidf_val_acc,
        "f1_at_train":tfidf_train_f1,
        "f1_at_val":tfidf_val_f1,
        "vectorizer":"TfidfVectorizer",
        "sublinear_tf":True})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_163024-tfidf_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run tfidf-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/tfidf_v1
wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.33375
wandb:   accuracy_at_val 0.3375
wandb:       f1_at_train 0.33152
wandb:         f1_at_val 0.33492
wandb:    map_at_3_train 0.52438
wandb:      map_at

In [21]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=10,window=1,min_count=1,seed=1,workers=1,epochs=10)

In [22]:
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [23]:
prompts=np.vstack(x_train['prompt'].apply(sentence_embedding))
option_cols=['A','B','C','D','E']
w2v_train_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_train[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_train_scores.append(scores)
w2v_train_scores=np.array(w2v_train_scores).T

In [24]:
val_prompts=np.vstack(x_val['prompt'].apply(sentence_embedding))
w2v_val_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_val[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(val_prompts,option_matrix))
    w2v_val_scores.append(scores)
w2v_val_scores=np.array(w2v_val_scores).T

In [25]:
w2v_train_labels=score_to_labels(w2v_train_scores)
w2v_train_map3=map_at_3(w2v_train_labels,y_train)
print(w2v_train_map3)

w2v_val_labels=score_to_labels(w2v_val_scores)
w2v_val_map3=map_at_3(w2v_val_labels,y_val)
print(w2v_val_map3)

0.3823958333333334
0.38583333333333336


In [26]:
w2v_train_acc,w2v_val_acc,w2v_train_f1,w2v_val_f1=support_metrics(w2v_train_scores,w2v_val_scores)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="word2vec-baseline-metrics",id="w2v_v1", resume="allow")
    wandb.log({
        "map_at_3_train":w2v_train_map3,   
        "map_at_3_val":w2v_val_map3,
        "accuracy_at_train":w2v_train_acc,
        "accuracy_at_val":w2v_val_acc,
        "f1_at_train":w2v_train_f1,
        "f1_at_val":w2v_val_f1,
        "embedding_model":"Word2Vec",
        "vector_size":20,
         "window":1,
         "min_count":1})
    wandb.finish()

wandb: setting up run w2v_v1
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_163050-w2v_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run word2vec-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/w2v_v1
wandb: updating run metadata
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb:         min_count ▁
wandb:       vector_size ▁
wandb:            window ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.185
wandb:   accuracy_at_val 0.175
wandb:   em

In [27]:
from transformers import AutoModel,AutoTokenizer
import torch

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer1=AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer2=AutoTokenizer.from_pretrained("roberta-base")
tokenizer3=AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

model1=AutoModel.from_pretrained("bert-base-uncased")
model2=AutoModel.from_pretrained("roberta-base")
model3=AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

model1.to(device)
model2.to(device)
model3.to(device)
model1.eval()
model2.eval()
model2.eval()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (14 outputs left out)

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [28]:
def get_embedding(text,tokenizer,model):
    if not text or not text.strip():
        return np.zeros(model.config.hidden_size)
    inputs=tokenizer(text,return_tensors='pt',max_length=128,truncation=True,padding=True)
    inputs={k:v.to(device) for k,v in inputs.items()}
    with torch.no_grad():
        outputs=model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

def get_scores(data,tokenizer,model):
    scores=[]
    prompts=np.vstack(data['prompt'].apply(lambda x:get_embedding(x,tokenizer,model)))
    for col in option_cols:
        options=np.vstack(data[col].apply(lambda x:get_embedding(x,tokenizer,model)))
        score=np.diag(cosine_similarity(prompts,options))
        scores.append(score)
    scores=np.array(scores).T
    labels=score_to_labels(scores)
    return scores,labels

bert_train_scores,bert_train_labels=get_scores(x_train,tokenizer1,model1)
bert_train=map_at_3(bert_train_labels,y_train)
bert_val_scores,bert_val_labels=get_scores(x_val,tokenizer1,model1)
bert_val=map_at_3(bert_val_labels,y_val)
roberta_train_scores,roberta_train_labels=get_scores(x_train,tokenizer2,model2)
roberta_train=map_at_3(roberta_train_labels,y_train)
roberta_val_scores,roberta_val_labels=get_scores(x_val,tokenizer2,model2)
roberta_val=map_at_3(roberta_val_labels,y_val)
st_train_scores,st_train_labels=get_scores(x_train,tokenizer3,model3)
st_train=map_at_3(st_train_labels,y_train)
st_val_scores,st_val_labels=get_scores(x_val,tokenizer3,model3)
st_val=map_at_3(st_val_labels,y_val)

print(bert_train)
print(bert_val)
print(roberta_train)
print(roberta_val)
print(st_train)
print(st_val) 

0.37302083333333336
0.38666666666666666
0.3094791666666667
0.3179166666666667
0.3930208333333334
0.39666666666666667


In [29]:
bert_train_acc,bert_val_acc,bert_train_f1,bert_val_f1=support_metrics(bert_train_scores,bert_val_scores)
roberta_train_acc,roberta_val_acc,roberta_train_f1,roberta_val_f1=support_metrics(roberta_train_scores,roberta_val_scores)
st_train_acc,st_val_acc,st_train_f1,st_val_f1=support_metrics(st_train_scores,st_val_scores)

In [30]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="BertEmbedding-metrics",id="bert_v1", resume="allow")
    wandb.log({
        "map_at_3_train":bert_train,   
        "map_at_3_val":bert_val,
        "accuracy_at_train":bert_train_acc,
        "accuracy_at_val":bert_val_acc,
        "f1_at_train":bert_train_f1,
        "f1_at_val":bert_val_f1,
        "embedding_model":"BERT",
        "max_length":128})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_163536-bert_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run BertEmbedding-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/bert_v1
wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb:        max_length ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.19937
wandb:   accuracy_at_val 0.2225
wandb:   embedding_model BERT
wandb:       f1_at_train 0.19936
wandb:         f1_at_val 0.22047
wandb:    map_at_3_train 0.37302
wandb:      map_at_3_val 0.38667
wandb:        max_length 128
wandb: 
wandb: 🚀 Vie

In [31]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="RobertaEmbedding-metrics",id="roberta_v1", resume="allow")
    wandb.log({
        "map_at_3_train":roberta_train,   
        "map_at_3_val":roberta_val,
        "accuracy_at_train":roberta_train_acc,
        "accuracy_at_val":roberta_val_acc,
        "f1_at_train":roberta_train_f1,
        "f1_at_val":roberta_val_f1,
        "embedding_model":"ROBERTA",
         "max_length":128})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_163537-roberta_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run RobertaEmbedding-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/roberta_v1
wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb:        max_length ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.16062
wandb:   accuracy_at_val 0.1725
wandb:   embedding_model ROBERTA
wandb:       f1_at_train 0.15687
wandb:         f1_at_val 0.16978
wandb:    map_at_3_train 0.30948
wandb:      map_at_3_val 0.31792
wandb:        max_length 128
wandb: 


In [32]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="SentenceTransformer-metrics",id="st_v1", resume="allow")
    wandb.log({
        "map_at_3_train":st_train,   
        "map_at_3_val":st_val,
        "accuracy_at_train":st_train_acc,
        "accuracy_at_val":st_val_acc,
        "f1_at_train":st_train_f1,
        "f1_at_val":st_val_f1,
        "embedding_model":"Sentence Transformer",
        "max_length":128})
    wandb.finish()

wandb: setting up run st_v1
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_163539-st_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run SentenceTransformer-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/st_v1
wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb:        max_length ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.22625
wandb:   accuracy_at_val 0.2425
wandb:   embedding_model Sentence Transformer...
wandb:       f1_at_train 0.22592
wandb:         f1_at_val 0.23861
wandb:    map_at_3_train 0.39302
wandb:      map_at_3_val 0.39667


In [33]:
from transformers import pipeline

classifier1=pipeline("zero-shot-classification",model="facebook/bart-large-mnli",device=0)

def zero_shot_apply(row,classifier,labels=True):
    options=[row[c] for c in option_cols]
    result=classifier(row['prompt'],candidate_labels=options,hypothesis_template="The answer to this question is {}.")
    ranked=result['labels'][:3]
    scores=result['scores']
    label_map={row[c]:c for c in option_cols}
    if labels:
        return " ".join([label_map[opt] for opt in ranked]),scores

z1_train_results=x_train.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()
z1_train_labels=[r[0] for r in z1_train_results]
z1_train_scores=[r[1] for r in z1_train_results]

z1_val_results=x_val.apply(lambda row: zero_shot_apply(row, classifier1, labels=True), axis=1).tolist()
z1_val_labels=[r[0] for r in z1_val_results]
z1_val_scores=[r[1] for r in z1_val_results]

z1_train_map3=map_at_3(z1_train_labels,y_train)
z1_val_map3=map_at_3(z1_val_labels,y_val)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [34]:
from transformers import pipeline

classifier2=pipeline("zero-shot-classification",model="cross-encoder/nli-MiniLM2-L6-H768",device=0)

z2_train_results=x_train.apply(lambda row:zero_shot_apply(row,classifier2,labels=True),axis=1).tolist()
z2_train_labels=[r[0] for r in z2_train_results]
z2_train_scores=[r[1] for r in z2_train_results]

z2_val_results=x_val.apply(lambda row:zero_shot_apply(row,classifier2,labels=True),axis=1).tolist()
z2_val_labels=[r[0] for r in z2_val_results]
z2_val_scores=[r[1] for r in z2_val_results]

z2_train_map3=map_at_3(z2_train_labels,y_train)
z2_val_map3=map_at_3(z2_val_labels,y_val)

config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-MiniLM2-L6-H768
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [35]:
print(z1_train_map3,z2_train_map3,z1_val_map3,z2_val_map3)

0.5716666666666667 0.5196875 0.5466666666666666 0.5316666666666666


In [36]:
from sklearn.preprocessing import StandardScaler

ss_tfidf=StandardScaler()
ss_w2v=StandardScaler()
train_final1=ss_tfidf.fit_transform(tfidf_train_scores)
train_final2=ss_w2v.fit_transform(w2v_train_scores)
val_final1=ss_tfidf.transform(tfidf_val_scores)
val_final2=ss_w2v.transform(w2v_val_scores)

In [37]:
def weighted_avg(final1,final2):
    best_score=0
    best_weights=[0.5,0.5]
    for first in np.arange(0,1.01,0.05):
        second=1-first
        if second<0:
            continue
        final=first*final1+second*final2
        labels=[]
        for score in final:
            scored=np.argsort(score)[::-1][:3]
            top_3=[chr(65+i) for i in scored]
            labels.append(" ".join(top_3)) 
        score_val = map_at_3(labels, y_train)
        if score_val>best_score:
            best_score=score_val
            best_weights=[first,second]
    return best_score,best_weights
best_score,best_weights=weighted_avg(train_final1,train_final2)
print(best_score,best_weights)

0.5522916666666666 [np.float64(0.8), np.float64(0.19999999999999996)]


In [38]:
ss_st=StandardScaler()
train_final3=ss_st.fit_transform(st_train_scores)
val_final3=ss_st.transform(st_val_scores)

best_score2,best_weights2=weighted_avg(train_final1,train_final3)
print(best_score2,best_weights2)

0.5552083333333333 [np.float64(0.65), np.float64(0.35)]


In [39]:
final_train1=best_weights[0]*train_final1+best_weights[1]*train_final2
final_val1=best_weights[0]*val_final1+best_weights[1]*val_final2

final_train_labels1=score_to_labels(final_train1)
final_val_labels1=score_to_labels(final_val1)

comb1_train_map3=map_at_3(final_train_labels1,y_train)
comb1_val_map3=map_at_3(final_val_labels1,y_val)

print(comb1_train_map3)
print(comb1_val_map3)

0.5522916666666666
0.5229166666666667


In [40]:
final_train2=best_weights2[0]*train_final1+best_weights2[1]*train_final3
final_val2=best_weights2[0]*val_final1+best_weights2[1]*val_final3

final_train_labels2=score_to_labels(final_train2)
final_val_labels2=score_to_labels(final_val2)

comb2_train_map3=map_at_3(final_train_labels2,y_train)
comb2_val_map3=map_at_3(final_val_labels2,y_val)

print(comb2_train_map3)
print(comb2_val_map3)

0.5552083333333333
0.5258333333333334


In [41]:
comb1_train_acc,comb1_val_acc,comb1_train_f1,comb1_val_f1=support_metrics(final_train1,final_val1)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="word2vec_and_tfidf-ensemble-metrics",id="tfidf+w2v_v1",resume="allow")
    wandb.log({
        "map_at_3_train":comb1_train_map3,   
        "map_at_3_val":comb1_val_map3,
        "accuracy_at_train":comb1_train_acc,
        "accuracy_at_val":comb1_val_acc,
        "f1_at_train":comb1_train_f1,
        "f1_at_val":comb1_val_f1,
        "embedding_model":"Word2Vec+TFIDF",
        "best_weights":best_weights})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_164121-tfidf+w2v_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run word2vec_and_tfidf-ensemble-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/tfidf+w2v_v1
wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.39125
wandb:   accuracy_at_val 0.38
wandb:   embedding_model Word2Vec+TFIDF
wandb:       f1_at_train 0.38682
wandb:         f1_at_val 0.37642
wandb:    map_at_3_train 0.55229
wandb:      map_at_3_val 0.52292
wandb: 
wandb: 🚀 View run word2vec_and_tfidf

In [42]:
comb2_train_acc,comb2_val_acc,comb2_train_f1,comb2_val_f1=support_metrics(final_train2,final_val2)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="SentenceTransformer_and_tfidf-ensemble-metrics",id="tfidf+st_v1",resume="allow")
    wandb.log({
        "map_at_3_train":comb2_train_map3,   
        "map_at_3_val":comb2_val_map3,
        "accuracy_at_train":comb2_train_acc,
        "accuracy_at_val":comb2_val_acc,
        "f1_at_train":comb2_train_f1,
        "f1_at_val":comb2_val_f1,
        "embedding_model":"Word2Vec+TFIDF",
        "best_weights":best_weights})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_164123-tfidf+st_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run SentenceTransformer_and_tfidf-ensemble-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/tfidf+st_v1
wandb: updating run metadata
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.36188
wandb:   accuracy_at_val 0.3375
wandb:   embedding_model Word2Vec+TFIDF
wandb:       f1_at_train 0.35672
wandb:        

In [43]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test['prompt']=test['prompt'].apply(clean_text)
test['A']=test['A'].apply(clean_text)
test['B']=test['B'].apply(clean_text)
test['C']=test['C'].apply(clean_text)
test['D']=test['D'].apply(clean_text)
test['E']=test['E'].apply(clean_text)

tfidf_test_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(test['prompt']+" "+test[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_test_scores.append(np.asarray(scores).flatten())
tfidf_test_scores=np.array(tfidf_test_scores).T

In [44]:
w2v_prompts=np.vstack(test['prompt'].apply(sentence_embedding))
test_w2v_scores=[]
for col in ['A','B','C','D','E']:
    option_matrix=np.vstack(test[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(w2v_prompts,option_matrix))
    test_w2v_scores.append(scores)

test_w2v_scores=np.array(test_w2v_scores).T

In [45]:
test_st_scores,st_test_labels=get_scores(test,tokenizer3,model3)

In [46]:
test_final1=ss_tfidf.transform(tfidf_test_scores)
test_final2=ss_w2v.transform(test_w2v_scores)
test_final3=ss_st.transform(test_st_scores)

In [47]:
final_test1=best_weights[0]*test_final1+best_weights[1]*test_final2
test_labels1=[]
for score in final_test1:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65+i) for i in scored]
    test_labels1.append(" ".join(top_3))

In [48]:
final_test2=best_weights2[0]*test_final1+best_weights2[1]*test_final3
test_labels2=[]
for score in final_test2:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65+i) for i in scored]
    test_labels2.append(" ".join(top_3))

In [49]:
z1_test_results=test.apply(lambda row:zero_shot_apply(row,classifier1,labels=True),axis=1).tolist()
z1_test_labels=[r[0] for r in z1_test_results]
z1_test_scores=[r[1] for r in z1_test_results]

In [50]:
submission=pd.DataFrame({'id':test['id'],'Prediction':z1_test_labels})
submission.to_csv('submission_model1_w2v.csv',index=False)
print("Model submission saved!")
print(submission.head())

Model submission saved!
   id Prediction
0   1      D E B
1   2      E B A
2   3      D A B
3   4      E A D
4   5      D C E
